In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-transpiler
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Транспіляція схем у хмарі за допомогою Qiskit Transpiler Service

> **Danger:** Станом на 18 липня 2025 року сервіс проходить міграцію для підтримки нової платформи IBM Quantum&reg; і наразі недоступний. Для AI-проходів можна використовувати [локальний режим](/guides/ai-transpiler-passes#run-the-ai-transpiler-passes-locally-or-on-the-cloud).
> 
>     Сервіс є бета-версією і може змінюватися.
>     Якщо у тебе є відгуки або ти хочеш зв'язатися з командою розробників, скористайся цим каналом [Qiskit Slack Workspace](https://qiskit.slack.com/archives/C06KF8YHUAU).

Qiskit Transpiler Service надає можливості транспіляції в хмарі. На додачу до локальних можливостей транспілятора Qiskit, твої завдання транспіляції можуть скористатися перевагами як хмарних ресурсів IBM Quantum, так і транспіляційних проходів на основі AI.

Qiskit Transpiler Service пропонує бібліотеку Python для безперебійної інтеграції цього сервісу та його можливостей у твої поточні патерни та робочі процеси Qiskit. Цей сервіс доступний лише для користувачів IBM Quantum Premium Plan, Flex Plan та On-Prem (через IBM Quantum Platform API) Plan.

<span id="install-transpiler-service"></span>
## Встановлення пакету qiskit-ibm-transpiler
Для використання Qiskit Transpiler Service встанови пакет `qiskit-ibm-transpiler`:

In [ ]:
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_transpiler.transpiler_service import TranspilerService

circuit = efficient_su2(101, entanglement="circular", reps=1)

cloud_transpiler_service = TranspilerService(
    backend_name="ibm_torino",
    ai="false",
    optimization_level=3,
)
transpiled_circuit = cloud_transpiler_service.run(circuit)

Пакет автоматично автентифікується за допомогою твоїх [облікових даних IBM Quantum Platform](/guides/cloud-setup), так само як [Qiskit Runtime керує цим](/guides/initialize-account):
- Змінна середовища: `QISKIT_IBM_TOKEN`
- Файл конфігурації `~/.qiskit/qiskit-ibm.json` (у розділі `default-ibm-quantum`).

*Примітка*: Цей пакет вимагає Qiskit SDK v1.X.

## Параметри транспіляції qiskit-ibm-transpiler
- `backend_name` (необов'язково, str) — Назва бекенду, як її очікує QiskitRuntimeService (наприклад, `ibm_torino`). Якщо задано, метод транспіляції використовує розкладку вказаного бекенду для операції транспіляції. Якщо задано будь-яку іншу опцію, що впливає на ці налаштування, наприклад `coupling_map`, налаштування `backend_name` перевизначаються.
- `coupling_map` (необов'язково, List[List[int]]) — Валідний список карти зв'язків (наприклад, [[0,1],[1,2]]). Якщо задано, метод транспіляції використовує цю карту зв'язків для операції транспіляції. Якщо визначено, перевизначає будь-яке значення, вказане для `target`.
- `optimization_level` (int) — Потенційний рівень оптимізації, що застосовується під час процесу транспіляції. Допустимі значення: [1,2,3], де 1 — найменша оптимізація (і найшвидша), а 3 — найбільша оптимізація (і найтриваліша).
- `ai` ("true", "false", "auto") — Чи використовувати можливості на основі AI під час транспіляції. Доступні можливості на основі AI можуть бути для транспіляційних проходів `AIRouting` або інших методів синтезу на основі AI. Якщо це значення `"true"`, сервіс застосовує різні транспіляційні проходи на основі AI залежно від запитуваного `optimization_level`. Якщо `"false"`, використовуються найновіші функції транспіляції Qiskit без AI. Нарешті, якщо `"auto"`, сервіс вирішує, чи застосовувати стандартні евристичні проходи Qiskit або проходи на основі AI, виходячи зі схеми.
- `qiskit_transpile_options` (dict) — Об'єкт словника Python, що може включати будь-яку іншу опцію, дійсну в [методі Qiskit `transpile()`](defaults-and-configuration-options). Якщо `qiskit_transpile_options` містить `optimization_level`, він ігнорується на користь `optimization_level`, вказаного як вхідний параметр. Якщо `qiskit_transpile_options` містить будь-яку опцію, не розпізнану методом Qiskit `transpile()`, бібліотека видає помилку.

Для отримання додаткової інформації про доступні методи `qiskit-ibm-transpiler` дивись [документацію API qiskit-ibm-transpiler](https://docs.quantum.ibm.com/api/qiskit-ibm-transpiler). Щоб дізнатися більше про API сервісу, дивись [документацію REST API Qiskit Transpiler Service.](https://docs.quantum.ibm.com/api/qiskit-transpiler-service-rest)

## Приклади
Наступні приклади демонструють, як транспілювати схеми за допомогою Qiskit Transpiler Service з різними параметрами.

1. Створи схему та виклич Qiskit Transpiler Service для транспіляції схеми з `ibm_torino` як `backend_name`, 3 як `optimization_level` та без використання AI під час транспіляції.

In [ ]:
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_transpiler.transpiler_service import TranspilerService

circuit = efficient_su2(101, entanglement="circular", reps=1)

cloud_transpiler_service = TranspilerService(
    backend_name="ibm_torino",
    ai="true",
    optimization_level=1,
)
transpiled_circuit = cloud_transpiler_service.run(circuit)

*Примітка*: можна використовувати лише пристрої `backend_name`, до яких є доступ з твоїм обліковим записом IBM Quantum. Крім `backend_name`, `TranspilerService` також приймає `coupling_map` як параметр.

2. Створи схожу схему та транспілюй її, запросивши можливості AI-транспіляції, встановивши прапорець `ai` у `True`:

In [ ]:
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_transpiler.transpiler_service import TranspilerService

circuit = efficient_su2(101, entanglement="circular", reps=1)

cloud_transpiler_service = TranspilerService(
    backend_name="ibm_torino",
    ai="auto",
    optimization_level=1,
)
transpiled_circuit = cloud_transpiler_service.run(circuit)

3. Створи схожу схему та транспілюй її, дозволивши сервісу самостійно вирішити, чи використовувати транспіляційні проходи на основі AI.